In [20]:
from gs_lib.gs_tools import *

def make_participants():
    m1, m2, m3, m4 = Man("m1"), Man("m2"), Man("m3"), Man("m4")
    w1, w2, w3, w4 = Woman("w1"), Woman("w2"), Woman("w3"), Woman("w4")
    return (m1, m2, m3, m4), (w1, w2, w3, w4)

def make_preferences(men, women):
    m1, m2, m3, m4 = men
    w1, w2, w3, w4 = women
    return PreferenceList({
        m1: [w1, w2, w3, w4],
        m2: [w2, w1, w4, w3],
        m3: [w3, w4, w1, w2],
        m4: [w4, w3, w2, w1],
        w1: [m4, m3, m2, m1],
        w2: [m3, m4, m1, m2],
        w3: [m2, m1, m4, m3],
        w4: [m1, m2, m3, m4],
    })


men, women = make_participants()
prefs = make_preferences(men, women)

In [21]:
print("=" * 70)
print("DEMO: BUILDING SIDES AND EXPRESSING MATCHINGS")
print("=" * 70)

# ── 1. CREATE THE TWO SIDES ──────────────────────────
print("\n1. CREATING MEN'S SIDE AND WOMEN'S SIDE")
print("-" * 40)

men = [Man("Adam"), Man("Bob"), Man("Charlie"), Man("David")]
women = [Woman("Eve"), Woman("Fiona"), Woman("Grace"), Woman("Helen")]

print(f"   Men:   {', '.join(m.id for m in men)}")
print(f"   Women: {', '.join(w.id for w in women)}")

# ── 2. EXPRESS PREFERENCES ───────────────────────────
print("\n2. EXPRESSING PREFERENCES")
print("-" * 40)

prefs = PreferenceList({
    # Men's preferences (1st > 2nd > 3rd > 4th)
    men[0]: [women[0], women[1], women[2], women[3]],  # Adam
    men[1]: [women[1], women[0], women[3], women[2]],  # Bob
    men[2]: [women[2], women[3], women[0], women[1]],  # Charlie
    men[3]: [women[3], women[2], women[1], women[0]],  # David
    
    # Women's preferences (1st > 2nd > 3rd > 4th)
    women[0]: [men[3], men[2], men[1], men[0]],        # Eve
    women[1]: [men[2], men[3], men[0], men[1]],        # Fiona
    women[2]: [men[1], men[0], men[3], men[2]],        # Grace
    women[3]: [men[0], men[1], men[2], men[3]],        # Helen
})

print("\n   Men's preferences:")
for m in men:
    ranked = " > ".join(w.id for w in prefs.get_preference(m))
    print(f"     {m.id}: {ranked}")

print("\n   Women's preferences:")
for w in women:
    ranked = " > ".join(m.id for m in prefs.get_preference(w))
    print(f"     {w.id}: {ranked}")

DEMO: BUILDING SIDES AND EXPRESSING MATCHINGS

1. CREATING MEN'S SIDE AND WOMEN'S SIDE
----------------------------------------
   Men:   Adam, Bob, Charlie, David
   Women: Eve, Fiona, Grace, Helen

2. EXPRESSING PREFERENCES
----------------------------------------

   Men's preferences:
     Adam: Eve > Fiona > Grace > Helen
     Bob: Fiona > Eve > Helen > Grace
     Charlie: Grace > Helen > Eve > Fiona
     David: Helen > Grace > Fiona > Eve

   Women's preferences:
     Eve: David > Charlie > Bob > Adam
     Fiona: Charlie > David > Adam > Bob
     Grace: Bob > Adam > David > Charlie
     Helen: Adam > Bob > Charlie > David


In [22]:
# ── 3. RUN GALE-SHAPLEY ──────────────────────────────
print("\n3. RUNNING GALE-SHAPLEY ALGORITHM")
print("-" * 40)

gs = GaleShapley(prefs)
man_opt, woman_opt = gs.find_both_optimal()

print(f"\n   Man-optimal matching:")
print(f"     {man_opt}")
print("     Satisfaction:")
for m in men:
    w = man_opt.get_partner(m)
    rank = prefs.get_rank(m, w) + 1
    print(f"       {m.id} → {w.id} (his #{rank} choice)")

print(f"\n   Woman-optimal matching:")
print(f"     {woman_opt}")
print("     Satisfaction:")
for w in women:
    m = woman_opt.get_partner(w)
    rank = prefs.get_rank(w, m) + 1
    print(f"       {w.id} → {m.id} (her #{rank} choice)")


3. RUNNING GALE-SHAPLEY ALGORITHM
----------------------------------------

   Man-optimal matching:
     Matching(pairs=[(Adam-Eve), (Bob-Fiona), (Charlie-Grace), (David-Helen)], unmatched_men=[], unmatched_women=[])
     Satisfaction:
       Adam → Eve (his #1 choice)
       Bob → Fiona (his #1 choice)
       Charlie → Grace (his #1 choice)
       David → Helen (his #1 choice)

   Woman-optimal matching:
     Matching(pairs=[(Adam-Helen), (Bob-Grace), (Charlie-Fiona), (David-Eve)], unmatched_men=[], unmatched_women=[])
     Satisfaction:
       Eve → David (her #1 choice)
       Fiona → Charlie (her #1 choice)
       Grace → Bob (her #1 choice)
       Helen → Adam (her #1 choice)


In [23]:
man_optimal = gs.find_stable_matching("men")
print(man_optimal)

Matching(pairs=[(Adam-Eve), (Bob-Fiona), (Charlie-Grace), (David-Helen)], unmatched_men=[], unmatched_women=[])


In [24]:
man_optimal = gs.find_stable_matching("women")
print(man_optimal)

Matching(pairs=[(Adam-Helen), (Bob-Grace), (Charlie-Fiona), (David-Eve)], unmatched_men=[], unmatched_women=[])


In [25]:
# Unequal two-sides:

m1u, m2u, m3u = Man("m1"), Man("m2"), Man("m3")
w1u, w2u = Woman("w1"), Woman("w2")

unequal = PreferenceList({
    m1u: [w1u, w2u],
    m2u: [w2u, w1u],
    m3u: [w1u],
    w1u: [m2u, m1u, m3u],
    w2u: [m1u, m2u],
})

gs_u = GaleShapley(unequal)
result = gs_u.find_stable_matching("men")
print(result)
# One man will end up in unmatched_men
print("Unmatched men:", result.unmatched_men)

Matching(pairs=[(m1-w1), (m2-w2)], unmatched_men=[m3], unmatched_women=[])
Unmatched men: frozenset({Man(m3)})


In [26]:
verifier = StabilityVerifier(prefs)
ok, reason, blocking = verifier.is_stable(man_optimal)
print(ok)          # True
print(reason)      # None
print(blocking)    # None

True
None
None


In [27]:
m1, m2, m3, m4 = Man("m1"), Man("m2"), Man("m3"), Man("m4")
w1, w2, w3, w4 = Woman("w1"), Woman("w2"), Woman("w3"), Woman("w4")

prefs = PreferenceList({
    m1: [w1, w2, w3, w4],
    m2: [w2, w1, w4, w3],
    m3: [w3, w4, w1, w2],
    m4: [w4, w3, w2, w1],
    w1: [m4, m3, m2, m1],
    w2: [m3, m4, m1, m2],
    w3: [m2, m1, m4, m3],
    w4: [m1, m2, m3, m4],
})

all_men   = {m1, m2, m3, m4}
all_women = {w1, w2, w3, w4}

verifier = StabilityVerifier(prefs)

In [28]:
reverse = Matching.from_dict(
    {m1: w4, m2: w3, m3: w2, m4: w1},
    all_men, all_women,
)
verifier.is_stable(reverse)

(True, None, None)

In [29]:


# --- Setup -------------------------------------------------------------------
m1, m2, m3, m4 = Man("m1"), Man("m2"), Man("m3"), Man("m4")
w1, w2, w3, w4 = Woman("w1"), Woman("w2"), Woman("w3"), Woman("w4")

prefs = PreferenceList({
    m1: [w1, w2, w3, w4],
    m2: [w2, w1, w4, w3],
    m3: [w3, w4, w1, w2],
    m4: [w4, w3, w2, w1],
    w1: [m4, m3, m2, m1],
    w2: [m3, m4, m1, m2],
    w3: [m2, m1, m4, m3],
    w4: [m1, m2, m3, m4],
})

all_men   = {m1, m2, m3, m4}
all_women = {w1, w2, w3, w4}

verifier = StabilityVerifier(prefs)


def check(label, matching):
    print("=" * 70)
    print(label)
    print("=" * 70)
    print(f"Matching: {matching}")
    ok, reason, blocking = verifier.is_stable(matching)
    if ok:
        print("Result:   ✓ STABLE")
    else:
        print("Result:   ✗ UNSTABLE")
        print(f"Reason:   {reason}")
        if blocking:
            man, woman = blocking
            print(f"Blocking: ({man.id}, {woman.id})")
    print()
    return ok


# --- CASE 1: identity matching ----------------------------------------------
identity = Matching.from_dict(
    {m1: w1, m2: w2, m3: w3, m4: w4},
    all_men, all_women,
)
check("CASE 1: Identity matching", identity)


# --- CASE 2: reverse matching -----------------------------------------------
reverse = Matching.from_dict(
    {m1: w4, m2: w3, m3: w2, m4: w1},
    all_men, all_women,
)
check("CASE 2: Reverse matching (m1-w4, m2-w3, m3-w2, m4-w1)", reverse)


# --- CASE 3: hand-picked, likely unstable -----------------------------------
handpicked = Matching.from_dict(
    {m1: w3, m2: w1, m3: w4, m4: w2},
    all_men, all_women,
)
check("CASE 3: Hand-picked (m1-w3, m2-w1, m3-w4, m4-w2)", handpicked)


# --- CASE 4: leave a man unmatched (valid when unequal, but here it's odd) --
partial = Matching.from_dict(
    {m1: w1, m2: w2, m3: w3, m4: None},   # m4 unmatched, w4 unmatched
    all_men, all_women,
)
check("CASE 4: m4 left unmatched", partial)


# --- CASE 5: individual-rationality violation -------------------------------
# m1's list has NO w4. Matching him to w4 should fail immediately.
irrational = Matching.from_dict(
    {m1: w4, m2: w1, m3: w2, m4: w3},
    all_men, all_women,
)
check("CASE 5: m1 matched to unacceptable w4", irrational)


# --- CASE 6: the actual man-optimal (should verify stable) ------------------
from stable_matching import GaleShapley
man_optimal = GaleShapley(prefs).find_stable_matching("men")
check("CASE 6: Man-optimal from Gale-Shapley", man_optimal)


# --- CASE 7: a matching that is stable but NOT produced by GS ---------------
# For this instance, we can search the lattice for a non-extreme node.
from stable_matching import StableMatchingLattice
lattice = StableMatchingLattice(prefs)
lattice.build_lattice()
all_stable = lattice.get_all_matchings()
non_extreme = [m for m in all_stable
               if m != man_optimal
               and m != GaleShapley(prefs).find_stable_matching("women")]
if non_extreme:
    check("CASE 7: A middle stable matching from the lattice", non_extreme[0])
else:
    print("CASE 7: no middle matching in this instance\n")

CASE 1: Identity matching
Matching: Matching(pairs=[(m1-w1), (m2-w2), (m3-w3), (m4-w4)], unmatched_men=[], unmatched_women=[])
Result:   ✓ STABLE

CASE 2: Reverse matching (m1-w4, m2-w3, m3-w2, m4-w1)
Matching: Matching(pairs=[(m1-w4), (m2-w3), (m3-w2), (m4-w1)], unmatched_men=[], unmatched_women=[])
Result:   ✓ STABLE

CASE 3: Hand-picked (m1-w3, m2-w1, m3-w4, m4-w2)
Matching: Matching(pairs=[(m1-w3), (m2-w1), (m3-w4), (m4-w2)], unmatched_men=[], unmatched_women=[])
Result:   ✓ STABLE

CASE 4: m4 left unmatched
Matching: Matching(pairs=[(m1-w1), (m2-w2), (m3-w3)], unmatched_men=[m4], unmatched_women=[w4])
Result:   ✗ UNSTABLE
Reason:   Blocking pair: (m4, w4)
Blocking: (m4, w4)

CASE 5: m1 matched to unacceptable w4
Matching: Matching(pairs=[(m1-w4), (m2-w1), (m3-w2), (m4-w3)], unmatched_men=[], unmatched_women=[])
Result:   ✗ UNSTABLE
Reason:   Blocking pair: (m3, w1)
Blocking: (m3, w1)



ModuleNotFoundError: No module named 'stable_matching'

In [30]:
partial = Matching.from_dict(
    {m1: w1, m2: w2, m3: w3, m4: None},   # m4 unmatched, w4 unmatched
    all_men, all_women,
)
verifier.is_stable(partial)

(False, 'Blocking pair: (m4, w4)', (Man(m4), Woman(w4)))